# VaaniRAG Offline Ingestion Pipeline - Colab Environment

This notebook is the reproducible environment to run and verify the offline ingestion pipeline.

In [ ]:
# CELL 1: Clone repository exactly once
!git clone https://github.com/yashvyas101/Vaani.git /content/Vaani || echo 'Already cloned'

In [ ]:
# CELL 2: cd into vaani-rag
%cd /content/Vaani/vaani-rag

In [ ]:
# CELL 3: install dependencies
!pip install -r requirements.txt

In [ ]:
# CELL 4: verify GPU
import torch
if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No CUDA GPU exists for the GPU ingestion benchmark.")

In [ ]:
# CELL 5: verify repository structure (Diagnostic Cell)
import os
import sys
sys.path.append(os.getcwd())

import ingestion.dataset_loader
import ingestion.pipeline

print(f"Current working directory: {os.getcwd()}")
print(f"dataset_loader file path: {ingestion.dataset_loader.__file__}")
print(f"pipeline file path:       {ingestion.pipeline.__file__}")

# Verify the source paths to prevent nested copy execution issues
expected_path = os.path.abspath(os.path.join(os.getcwd(), 'ingestion', 'dataset_loader.py'))
actual_path = os.path.abspath(ingestion.dataset_loader.__file__).replace('.pyc', '.py')

print(f"Expected source path:     {expected_path}")
print(f"Actual source path:       {actual_path}")

assert actual_path == expected_path, "Error: Executing wrong nested copy of the project!"
print("Repository structure verification: SUCCESS")

In [ ]:
# CELL 6: inspect dataset
!python scripts/inspect_dataset.py

In [ ]:
# CELL 7: test BGE-M3
!python scripts/test_embedding.py

In [ ]:
# CELL 8: run extraction-only smoke test
import sys
import os
sys.path.append(os.getcwd())
from ingestion.dataset_loader import load_dataset_stream
from ingestion.passage_extractor import extract_passages_from_row

print("Loading dataset default stream...")
stream = load_dataset_stream(split='train')
print("Extracting first 3 rows...")
for idx, row in enumerate(stream):
    if idx >= 3:
        break
    passages = list(extract_passages_from_row(row, idx))
    print(f"\nRow {idx} metadata: target_lang='{row.get('target_lang')}', source_lang='{row.get('source_lang')}'")
    print(f"Extracted {len(passages)} passages:")
    for p in passages:
        print(f"  - [{p.language}] ID={p.passage_id[:30]}... Text={p.text[:60]}...")
print("\nExtraction smoke test PASSED")

In [ ]:
# CELL 9: run 10-row dry run
!python -m ingestion.pipeline --languages en,hi,mr --max-rows 10 --strategy adaptive --dry-run

In [ ]:
# CELL 10: run 100-row dry run
!python -m ingestion.pipeline --languages en,hi,mr --max-rows 100 --strategy adaptive --dry-run

In [ ]:
# CELL 11: optional Pinecone connection test
import os
try:
    from google.colab import userdata
    api_key = userdata.get('PINECONE_API_KEY')
    if api_key:
        os.environ['PINECONE_API_KEY'] = api_key
        print("Loaded Pinecone API key from Colab secrets.")
        from ingestion.pinecone_client import get_pinecone_client
        pc = get_pinecone_client()
        indexes = pc.list_indexes()
        print(f"Successfully connected to Pinecone. Existing Indexes: {[idx.name for idx in indexes]}")
    else:
        print("PINECONE_API_KEY not found in Colab secrets. Skipping connection test.")
except Exception as e:
    print(f"Pinecone connection test skipped: {e}")

In [ ]:
# CELL 12: optional Pinecone upload
# Run this cell only if you wish to upload vectors to Pinecone Cloud (requires PINECONE_API_KEY set)
!python -m ingestion.pipeline --languages en,hi,mr --max-rows 100 --strategy adaptive --upload